## Environment setup 

In [112]:
%pip install accelerate bitsandbytes google-cloud-aiplatform peft trl scikit-learn tokenizers torch transformers unsloth evaluate python-dotenv sentencepiece protobuf matplotlib

Note: you may need to restart the kernel to use updated packages.


## import library

In [113]:
import os
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments
import evaluate
from datasets import load_dataset
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from huggingface_hub import login
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import Subset

## Huggingface setup

In [114]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token: 
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("HUGFACE_TOKEN not found in environment variables. Please set it to access Hugging Face resources.")

Successfully logged in to Hugging Face.


## Load Dataset

In [ ]:
from torchvision import datasets

train_data = datasets.EMNIST(root='./data', split='byclass', train=True, download=True)
test_data = datasets.EMNIST(root='./data', split='byclass', train=False, download=True)
train_data, test_data

(<torch.utils.data.dataset.Subset at 0x76e7c37e44a0>,
 <torch.utils.data.dataset.Subset at 0x76e9eb557230>)

## visualize data

In [116]:

def count_labels(dataset):
    labels = [label for _, label in dataset] 
    counter = Counter(labels)
    return [counter.get(i, 0) for i in range(62)]

train_counts = count_labels(train_data)
test_counts  = count_labels(test_data)
class_names = train_data.classes
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

axes[0].bar(range(62), train_counts, color='steelblue', alpha=0.85)
axes[0].set_title('Train set - Số mẫu mỗi lớp', fontsize=13)
axes[0].set_ylabel('Số lượng')
axes[0].set_xticks(range(62))
axes[0].set_xticklabels(class_names, fontsize=6, rotation=90)

axes[1].bar(range(62), test_counts, color='coral', alpha=0.85)
axes[1].set_title('Test set - Số mẫu mỗi lớp', fontsize=13)
axes[1].set_xlabel('Lớp')
axes[1].set_ylabel('Số lượng')
axes[1].set_xticks(range(62))
axes[1].set_xticklabels(class_names, fontsize=6, rotation=90)

plt.tight_layout()
plt.show()

AttributeError: 'Subset' object has no attribute 'classes'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
class EMNISTWrapper(Dataset):
    def __init__(self, original_dataset, processor):
        self.dataset = original_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = img.convert("RGB").resize((224, 224))
        encoded = self.processor(images=img, return_tensors="pt")
        encoded = {k: v.squeeze(0) for k, v in encoded.items()}
        encoded["labels"] = label
        return encoded


### Model

In [ ]:
# model_name = "facebook/convnext-tiny-224"
# model_name = "Peadar/rt-detr-v2-s"
model_name = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(model_name, use_fast=True)

train_data = EMNISTWrapper(train_data, processor)
test_data  = EMNISTWrapper(test_data, processor)

model1 = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=len(train_data.dataset.classes),
    id2label={i: label for i, label in enumerate(train_data.dataset.classes)},
    label2id={label: i for i, label in enumerate(train_data.dataset.classes)},
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1)
model1


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

### Evaluation Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

### Training Arguments and Trainer

In [ ]:
train_labels = [label for _, label in train_data.dataset]
unique_classes = np.unique(train_labels)
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:")
for label, weight in zip(unique_classes, class_weights.tolist()):
    print(f"{label}: {weight:.4f}")

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        target_labels = inputs.pop("labels", None)
        if target_labels is None:
            target_labels = inputs.pop("label")

        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, len(unique_classes)), target_labels.view(-1))
        return (loss, outputs) if return_outputs else loss

Class weights:
0: 0.3255
1: 0.2933
2: 0.3291
3: 0.3203
4: 0.3357
5: 0.3583
6: 0.3288
7: 0.3148
8: 0.3316
9: 0.3326
10: 1.7570
11: 2.9028
12: 1.1152
13: 2.4676
14: 2.2815
15: 1.2260
16: 4.4724
17: 3.5714
18: 0.9423
19: 2.9923
20: 4.5612
21: 2.2177
22: 1.2505
23: 1.3666
24: 0.4506
25: 1.3486
26: 4.3213
27: 2.2190
28: 0.5421
29: 1.1463
30: 0.8933
31: 2.4276
32: 2.3977
33: 4.0624
34: 2.3734
35: 4.1677
36: 1.1220
37: 2.1820
38: 3.9443
39: 1.1061
40: 0.4570
41: 4.3955
42: 3.0532
43: 1.2883
44: 4.1310
45: 5.9372
46: 4.5191
47: 0.7349
48: 4.2559
49: 0.9859
50: 4.0949
51: 4.5984
52: 3.7598
53: 0.7981
54: 4.1708
55: 0.6164
56: 3.9777
57: 3.8684
58: 4.1739
59: 3.9890
60: 4.7598
61: 4.1310


In [ ]:
repo_name = "vit-emnist-byclass"
training_args = TrainingArguments(
    output_dir=repo_name,
    eval_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=True,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)
trainer = CustomTrainer(
    model=model1,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_2443/2699418151.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


### Fine-tuning

In [ ]:
print("Training...")
trainer.train()

Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 